In [1]:
import pandas as pd
import re
from langchain.text_splitter import RecursiveCharacterTextSplitter

# Step 1: Load the CSV
df = pd.read_csv(r"C:\Users\hp\Downloads\merged_cleaned_text_PaddleOCR2.csv")

# 🔍 Check what's inside
print(df.head())  

     filename                                               text
0     0_0.txt  [ WIKIPEDIA Donate Create account Log in. The ...
1     0_1.txt  [ Animation [edit] Year Title Role Notes 1997 ...
2    0_10.txt  [ This page was last edited on 20 April 2025, ...
3   0_100.txt  [ Billy Joel's parents met in the late 1930s a...
4  0_1000.txt  [ Other early instances of women's suffrage in...


In [3]:
def clean_text(text):
    text = str(text).strip()
    text = re.sub(r'\s+', ' ', text)              # Replace multiple spaces/newlines
    text = re.sub(r'[^\x00-\x7F]+', '', text)     # Remove non-ASCII characters
    return text

# Apply cleaning to each row in 'Extracted Text'
cleaned_texts = df["text"].dropna().apply(clean_text)

In [4]:
combined_text = " ".join(cleaned_texts.tolist())

In [7]:
# Step 4: Chunk using RecursiveCharacterTextSplitter
from langchain.text_splitter import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100
)

chunks = splitter.split_text(combined_text)

for i, chunk in enumerate(chunks[:3]):
    print(f"\nChunk {i+1}:\n{chunk}")
    import pandas as pd

# Create a DataFrame from the chunks
df_chunks = pd.DataFrame({
    "chunk_id": [f"chunk_{i+1}" for i in range(len(chunks))],
    "text": chunks
})

# Save to CSV locally
df_chunks.to_csv(r"C:\Users\hp\Downloads\textchunks_output.csv", index=False, encoding="utf-8")

print("✅ Chunks saved to textchunks_output.csv")



Chunk 1:
[ WIKIPEDIA Donate Create account Log in. The Free Encyclopedia Marieve Herington 5 languages ArticleTalk Read Edit View history Tools. From Wikipedia, the free encyclopedia Marieve Herington (born February 22, 1988) is a Canadian Marieve Herington actress who has appeared in recurring roles on How I Met Born February 22, 1988 (age 37)[1] Your Mother, Good Luck Charlie and Ever After High. She. Oakville, Ontario, Canada[1] provides the voice of Tilly Green on the Disney Channel show. Occupations Actress singer Big City Greens as well as voicing animated lead characters in Years active1996-present Delilah & Julius and Pearlie. In addition to that, she voiced. Website marieveherington.com Celestia Ludenburg in the popular anime video game Danganronpa: Trigger Happy Havoc, Otter in Franklin and Fifi La Fume in Tiny Toons Looniversity. At the age of 12, she began singing in major public performances.Since the age of 16, she has been fronting her own jazz ensembles.Currently, she 

In [6]:
pip install sentence-transformers


Defaulting to user installation because normal site-packages is not writeableNote: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
#Generate Embeddings for Your Chunks, firstly loading chunking file.
import pandas as pd

# Load your chunked file
df = pd.read_csv(r"C:\Users\hp\Downloads\textchunks_output.csv")

# Make sure 'text' column exists
print(df.columns)
chunks = df["text"].dropna().tolist()


Index(['chunk_id', 'text'], dtype='object')


In [10]:
# Load Embedding Model & Generate Embeddings
from sentence_transformers import SentenceTransformer

# Load the model
model = SentenceTransformer('all-MiniLM-L6-v2')

# Generate embeddings
embeddings = model.encode(chunks, show_progress_bar=True)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

C:\Users\hp\AppData\Roaming\Python\Python312\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\hp\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/751 [00:00<?, ?it/s]

In [12]:
df["embedding"] = embeddings.tolist()

# Save to a new CSV
df.to_csv(r"C:\Users\hp\Downloads\recursive_chunks_with_embeddings.csv", index=False)
print("✅ Saved as 'recursive_chunks_with_embeddings.csv'")


✅ Saved as 'recursive_chunks_with_embeddings.csv'
